# Physical analysis

In [1]:
import fesslix as flx
flx.load_engine()

Random Number Generator: MT19937 - initialized with rand()=843731345;
Random Number Generator: MT19937 - initialized with 1000 initial calls.


0


```{todo}
Write this section.
```


## dew point and humidity

### saturation vapor pressure from temperature

```{eval-rst}
.. function:: flx.phys_temp2svp

    Syntax:
        ``flx.phys_temp2svp(temp)``

    Description:
        Evaluates the saturation vapor pressure as a function of temperature `temp`.
        
    :param temp: temperature
    :type temp: float
    :returns: saturation vapor pressure [hPa]
    :rtype: float
```

**Example**:

For a temperature of 22°C, the saturation vapor pressure is:

In [2]:
print(f"{flx.phys_temp2svp(22.):.2f} hPa")

26.37 hPa


### vapor pressure

Vapor pressure is evaluated by multiplying the saturation vapor pressure with the relative humidity.
Note that vapor pressure is the **primary conserved variable** in steady-state building physics.

**Example**:

For a temperature of 22°C and 40% humidity, the vapor pressure is:

In [3]:
vp = 0.4 * flx.phys_temp2svp(22.)
print(f"{vp:.2f} hPa")

10.55 hPa


### dew point from temperature and relative humidity

```{eval-rst}
.. function:: flx.phys_dewpoint

    Syntax:
        ``flx.phys_dewpoint(temp, phi)``

    Description:
        Evaluates the dew point as a function of temperature `temp` and humidity `phi`.
        
    :param temp: temperature
    :type temp: float
    :param phi: humidity
    :type phi: float
    :rtype: float
    
```

**Example**:

For a temperature of 22°C and 40% humidity, the dew point is:

In [4]:
dew_point = flx.phys_dewpoint(22.,0.4 )
print(f"{dew_point:.1f} °C")

7.8 °C


### temperature from dew point and relative humidity

```{eval-rst}
.. function:: flx.phys_tauphi2temp

    Syntax:
        ``flx.phys_tauphi2temp(temp_dewp, phi)``

    Description:
        Evaluates the temperature associated with a given dew point `temp_dewp` and humidity `phi`.
        
    :param temp_dewp: dew point (temperature)
    :type temp_dewp: float
    :param phi: humidity
    :type phi: float
    :rtype: float
    
```

**Example**:

If the dew point is at 7.8°C and we have a humidity of 40%, the associated temperature is:

In [5]:
temp = flx.phys_tauphi2temp(dew_point,0.4 )
print(f"{temp:.1f} °C")

22.0 °C


### relative humidity from dew point and temperature

```{eval-rst}
.. function:: flx.phys_tauptemp2phi

    Syntax:
        ``flx.phys_tauptemp2phi(temp_dewp, temp)``

    Description:
        Evaluates the humidity associated with a given dew point `temp_dewp` and temperature `temp`.
        
    :param temp_dewp: dew point (temperature)
    :type temp_dewp: float
    :param temp: temperature
    :type temp: float
    :rtype: float
    
```

**Example**:

If the dew point is at 7.8°C and the temperature is 22°C, the humidity is:

In [6]:
phi = flx.phys_tauptemp2phi(dew_point, 22. )
print(f"{phi*100:.0f}%")

40%


### absolute humidity from temperature and relative humidity

```{eval-rst}
.. function:: flx.phys_abs_humidity

    Syntax:
        ``flx.phys_abs_humidity(temp, phi)``

    Description:
        Evaluates the absolute humidity as a function of temperature `temp` and humidity `phi`.
        
    :param temp: temperature
    :type temp: float
    :param phi: humidity
    :type phi: float
    :returns: absolute humidity in [g/m³]
    :rtype: float
    
```

**Example**:

For a temperature of 22°C and 40% humidity, the absolute humidity is:

In [7]:
abs_hum = flx.phys_abs_humidity(22.,0.4 )
print(f"{abs_hum:.1f} g/m³")

7.7 g/m³


### Application examples

#### Maximum humidity to avoid mold (on walls)

In [8]:
## surface temperature (of critical element)
t_s = 9. ## [°C]

## temperature at dew point
t_dew = flx.phys_dewpoint(t_s,0.8 )  ## above 80% relative humidity at the critical element, it becomes critical 
print(f"{t_dew = :.1f} °C")
## vapor pressure
svp = flx.phys_temp2svp(t_s)
vp = 0.8 * svp
print(f"{vp = :.2f} hPa")
## absolute humidity at surface
t_s_abs = flx.phys_abs_humidity(t_s,0.8 )
print(f"{t_s_abs = :.1f} g/m³")
print()

## mold becomes likely if for the specified temperatures, humidity is exceeded
print("room temp    rel. humidity   vapor pressure    abs. humidity")
for temp in [19.,20.,21.,22.,23.]:
    phi_of_temp = flx.phys_tauptemp2phi(t_dew, temp )
    print( f"{temp:.0f}°C,        {phi_of_temp*100:.0f}%,            {phi_of_temp*flx.phys_temp2svp(temp):.2f} hPa          {flx.phys_abs_humidity(temp,phi_of_temp ):.1f} g/m³" )
    

t_dew = 5.7 °C
vp = 9.17 hPa
t_s_abs = 7.0 g/m³

room temp    rel. humidity   vapor pressure    abs. humidity
19°C,        42%,            9.17 hPa          6.8 g/m³
20°C,        39%,            9.17 hPa          6.8 g/m³
21°C,        37%,            9.17 hPa          6.8 g/m³
22°C,        35%,            9.17 hPa          6.7 g/m³
23°C,        33%,            9.17 hPa          6.7 g/m³


#### Airing of a room (minimum humidity that can theoretically be achieved)

In [23]:
## outside temperature
t_out = 5.2  ## [°C]
## outside humidity
phi_out = 0.9
## inside temperature
t_in_lst = [19.,20.,21.,22.,23.]

## vapor pressure
vp = phi_out * flx.phys_temp2svp(t_out)
print(f"{vp      = :.2f} hPa")

## temperature at dew point
t_dew = flx.phys_dewpoint(t_out,phi_out )
print(f"{t_dew   = :.1f}°C")

## absolute humidity
abs_hum = flx.phys_abs_humidity(t_out,phi_out )
print(f"{abs_hum = :.1f} g/m³")

print()
## minimum humidity (inside) that can be (theoretically) achieved for the specified temperature
print("temp  rel. humidity    vapor pressure   abs. humidity")
for t_in in t_in_lst:
    phi_in = flx.phys_tauptemp2phi(t_dew, t_in )
    print( f"{t_in:.0f}°C, {phi_in*100:.0f}%,             {phi_in*flx.phys_temp2svp(t_in):.2f} hPa         {flx.phys_abs_humidity(t_in,flx.phys_tauptemp2phi(t_dew, t_in ) ):.3f} g/m³" )
    

vp      = 7.96 hPa
t_dew   = 3.7°C
abs_hum = 6.2 g/m³

temp  rel. humidity    vapor pressure   abs. humidity
19°C, 36%,             7.96 hPa         5.901 g/m³
20°C, 34%,             7.96 hPa         5.881 g/m³
21°C, 32%,             7.96 hPa         5.861 g/m³
22°C, 30%,             7.96 hPa         5.841 g/m³
23°C, 28%,             7.96 hPa         5.821 g/m³
